In [45]:
from fredapi import Fred
import yfinance as yf
import pandas_datareader.data as web

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import linregress
from itertools import product
from typing import List, Tuple, Dict
import numpy as np
import pandas as pd
from typing import List,Optional,Dict,Any
import matplotlib.pyplot as plt
from openbb import obb

In [74]:
class DataHandler:
    """Data handler class for loading and processing data."""

    def __init__(
        self,
        symbol: str,
        start_date: Optional[str] = None,
        end_date: Optional[str] = None,
        provider: str = "fmp",
    ):
        """Initialize the data handler."""
        self.symbol = symbol.upper()
        self.start_date = start_date
        self.end_date = end_date
        self.provider = provider

    def load_data(self) -> pd.DataFrame | dict[str, pd.DataFrame]:
        """Load equity data."""
        data = obb.equity.price.historical(
            symbol=self.symbol,
            start_date=self.start_date,
            end_date=self.end_date,
            provider=self.provider,
        ).to_df()

        if "," in self.symbol:
            data = data.reset_index().set_index("symbol")
            return {symbol: data.loc[symbol] for symbol in self.symbol.split(",")}

        return data

    def load_data_from_csv(self, file_path) -> pd.DataFrame:
        """Load data from CSV file."""
        return pd.read_csv(file_path, index_col="date", parse_dates=True)
    

class Strategy:
    """Base class for trading stretegies"""

    def __init__(self, indicators: dict, signal_logic:Any):
        self.indicators = indicators
        self.signal_logic = signal_logic
    
    def generate_signals(
        self, data: pd.DataFrame | dict[str, pd.DataFrame]
        ) -> pd.DataFrame | dict[str, pd.DataFrame]:
        """Genrate signals based on the strategy'y indicators and signal logic"""
        if isinstance(data,dict):
            for _,asset_data in data.items():
                self._apply_strategy(asset_data)
        else:
            self._apply_strategy(data)
        return data
    
    def _apply_strategy(self, df: pd.DataFrame) -> None:
        """Apply strategy to a dataframe"""

        for name, indicator in self.indicators.items():
            df[name] = indicator(df)

        df['signal'] = df.apply(self.signal_logic, axis=1)
        df["positions"] = df['signal'].diff().fillna(0)


class Backtester:
    def __init__(
        self,
        initial_capital: float = 100000.0,
        commission_pct: float = 0.005,
        commission_fixed: float = 1.0,
        trade_fraction: float = 0.3,
    ):
        self.initial_capital = initial_capital
        self.commission_pct = commission_pct
        self.commission_fixed = commission_fixed
        self.trade_fraction = trade_fraction

        self.assets_data: Dict = {}
        self.portfolio_history: Dict = {}
        self.daily_portfolio_values: List[float] = []
        self.dates: List[pd.Timestamp] = []
        self.trade_log: Dict[str, List[dict]] = {}

    def execute_trade(self, asset: str, signal: int, price: float, date: pd.Timestamp = None, per_trade_value: float = None) -> None:
        if signal > 0 and self.assets_data[asset]["positions"] == 0:
    
            trade_value = per_trade_value if per_trade_value is not None else (
                self.assets_data[asset]["cash"] + self.assets_data[asset]["position_value"]
            ) * self.trade_fraction

            trade_value = min(trade_value, self.assets_data[asset]["cash"])
            commission = self.calculate_commission(trade_value)
            shares_to_buy = (trade_value - commission) / price

            if shares_to_buy > 0:
                self.assets_data[asset]['positions'] += shares_to_buy
                self.assets_data[asset]['cash'] -= trade_value

                self.trade_log.setdefault(asset, []).append({
                    "date": date,
                    "type": "BUY",
                    "shares": shares_to_buy,
                    "price": price,
                    "value": trade_value,
                    "commission": commission,
                    "cash_remaining": self.assets_data[asset]['cash'],
                })

        elif signal < 0 and self.assets_data[asset]['positions'] > 0:
            shares_to_sell = self.assets_data[asset]["positions"]
            trade_value = shares_to_sell * price
            commission = self.calculate_commission(trade_value)

            self.assets_data[asset]["cash"] += trade_value - commission
            self.assets_data[asset]["positions"] = 0

            self.trade_log.setdefault(asset, []).append({
                "date": date,
                "type": "SELL",
                "shares": shares_to_sell,
                "price": price,
                "value": trade_value,
                "commission": commission,
                "cash_remaining": self.assets_data[asset]['cash'],
            })

    def update_portfolio(self, pair: str, price: float) -> None:
        self.assets_data[pair]["position_value"] = (self.assets_data[pair]["positions"] * price)
        self.assets_data[pair]["total_value"] = (
            self.assets_data[pair]["cash"] + self.assets_data[pair]["position_value"]
        )
        self.portfolio_history[pair].append(self.assets_data[pair]["total_value"])

    def backtest(self, data: pd.DataFrame | dict[str, pd.DataFrame]):
        unique_pairs = data['pair'].unique()
        for pair in unique_pairs:
            self.assets_data[pair] = {
                "cash": self.initial_capital / len(unique_pairs),
                "positions": 0,
                "position_value": 0,
                "total_value": 0,
            }
            self.portfolio_history[pair] = []

        grouped = data.groupby(data.index)

        for date, group in grouped:
            total_portfolio_value = 0

            # Determine which buy signals are valid (no open position)
            eligible_buys = group[
                (group['signal'] > 0) & (group['pair'].apply(lambda p: self.assets_data[p]['positions'] == 0))
            ]

            top_signals = eligible_buys.head(20)  # max 20 positions per day
            num_trades = len(top_signals)

            for _, row in group.iterrows():
                pair = row['pair']
                price = row['price']
                signal = row['signal']

                # Calculate how much to allocate per trade (split across valid buys)
                if signal > 0 and pair in top_signals['pair'].values and num_trades > 0:
                    total_value = self.assets_data[pair]["cash"] + self.assets_data[pair]["position_value"]
                    per_trade_value = total_value * self.trade_fraction / num_trades
                    self.execute_trade(pair, signal, price, date, per_trade_value=per_trade_value)
                elif signal < 0:
                    self.execute_trade(pair, signal, price, date)

                self.update_portfolio(pair, price)
                total_portfolio_value += self.assets_data[pair]["total_value"]

            self.daily_portfolio_values.append(total_portfolio_value)
            self.dates.append(date)


    def calculate_performance(self, plot: bool = True):
        if not self.daily_portfolio_values:
            print("No data.")
            return

        portfolio_series = pd.Series(self.daily_portfolio_values, index=self.dates)
        daily_returns = portfolio_series.pct_change().dropna()

        final_value = portfolio_series.iloc[-1]
        total_return = (final_value - self.initial_capital) / self.initial_capital
        annualized_return = (1 + total_return) ** (252 / len(portfolio_series)) - 1
        volatility = daily_returns.std() * np.sqrt(252)
        sharpe = (annualized_return - 0.02) / volatility if volatility > 0 else np.nan

        print(f"Final Value: {final_value:.2f}")
        print(f"Total Return: {total_return:.2%}")
        print(f"Annualized Return: {annualized_return:.2%}")
        print(f"Volatility: {volatility:.2%}")
        print(f"Sharpe Ratio: {sharpe:.2f}")

        if plot:
            self.plot_performance(portfolio_series, daily_returns)

    def plot_performance(self, portfolio_values, daily_returns):
        fig, axs = plt.subplots(2, 1, figsize=(10, 6))
        axs[0].plot(portfolio_values)
        axs[0].set_title("Portfolio Value")
        axs[1].plot(daily_returns, color="orange")
        axs[1].set_title("Daily Returns")
        plt.tight_layout()
        plt.show()


In [75]:
### Current Tickers ###
mineral_stocks = [
    'GC=F',  # Gold
    'HG=F',  # Copper
    'PA=F',  # Palladium
    'PL=F',  # Platinum
    'SI=F', # Silver
    'CL=F', 
    'ZC=F',   #corn
    'LE=F', #cattle
    'HE=F', #hog
    'SB=F' #sugar
]

growth_stocks = [
    "NVDA",  # NVIDIA Corporation
    "MRVL",  # Marvell Technology Inc.
    "FTNT",  # Fortinet Inc.
    "AMD",   # Advanced Micro Devices, Inc.
    "CRM",   # Salesforce Inc.
    "ADBE",  # Adobe Inc.
    "ZM",    # Zoom Video Communications Inc.
    "SHOP",  # Shopify Inc.
    "SNAP",  # Snap Inc.
    "NET",   # Cloudflare, Inc.
    "TWLO",  # Twilio Inc.
    "MDB",   # MongoDB, Inc.
    "RBLX",  # Roblox Corporation
    "SNOW",  # Snowflake Inc.
    "PINS",  # Pinterest Inc.
    "TTD",   # The Trade Desk
    "DOCU",  # DocuSign, Inc.
    'SLAB',  # Silicon Laboratories Inc.
]

value_stocks = [
    "AAPL",  # Apple Inc.
    "MSFT",  # Microsoft Corporation
    "INTC",  # Intel Corporation
    "IBM",   # International Business Machines Corporation
    "ORCL",  # Oracle Corporation
    "CSCO",  # Cisco Systems, Inc.
    "HPE",   # Hewlett Packard Enterprise Co.
    "QCOM",  # Qualcomm Incorporated
    "TXN",   # Texas Instruments Incorporated
    "AVGO",  # Broadcom Inc.
    "MU",    # Micron Technology Inc.
    "LRCX",  # Lam Research Corporation
    "STX",   # Seagate Technology Holdings PLC
    "WDC",   # Western Digital Corporation
    "ADI",   # Analog Devices, Inc.
    "AMAT",  # Applied Materials, Inc.
    "MSI",   # Motorola Solutions, Inc.
    "VZ",    # Verizon Communications Inc.
    "TMUS" ,  # T-Mobile US, Inc.
    "DE",     # Deere & Co (agricultural tech & equipment)
    "LULU",   # Lululemon Athletica (consumer discretionary)
    #"FISV",   # Fiserv (fintech)
    "VRTX",   # Vertex Pharmaceuticals (biotech)
    "ETN",    # Eaton Corp (industrial electrical systems)
    "CTAS",   # Cintas Corp (B2B services)
    "NOC",    # Northrop Grumman (defense)
    "HUM",    # Humana Inc (healthcare)
    "REGN",   # Regeneron Pharmaceuticals
    "CMI",    # Cummins Inc (industrial engines)
    "SRE",    # Sempra Energy (utilities + infrastructure)
    "TT",     # Trane Technologies (climate control systems)
    "DOV",    # Dover Corp (industrial manufacturing)
    "APD",    # Air Products & Chemicals
    "KEYS"    # Keysight Technologies (testing & measurement)

]

market_indices = [
    "^DJI",     # Dow Jones Industrial Average (United States)
    "^GSPC",    # S&P 500 (United States)
    "^IXIC",    # NASDAQ Composite (United States)
    "^N225",    # Nikkei 225 (Japan)
    "^FTSE",    # FTSE 100 (United Kingdom)
    "^GDAXI",   # DAX (Germany)
    "^FCHI",    # CAC 40 (France)
    "HSI",      # Hang Seng Index (Hong Kong)
    "000001.SS",# Shanghai Composite Index (China)
    "^BSESN",   # SENSEX (India)
    "^NSEI",    # Nifty 50 (India)
    "^KS11",    # KOSPI (South Korea)
    "^AORD",    # All Ordinaries (Australia)
    "^BVSP",    # Bovespa (Brazil)
    "^MERV",    # MERVAL (Argentina)
    "^TWII",    # TAIEX (Taiwan)
    "^STI",     # Straits Times Index (Singapore)
    "^JKSE",    # Jakarta Composite Index (Indonesia)
    
]


In [76]:
start_date = "2022-01-01"


### Download Mineral Data ###
minerals = yf.download(mineral_stocks,start=start_date,auto_adjust=True,progress=False)["Close"]
minerals = minerals.dropna()
# minerals.columns = [(x[1] +' '+ x[0]).strip() for x in minerals.columns.to_list()]



## Download Closing Price of Growth and Value Stocks in One DataFrame ###
all_stocks = growth_stocks + value_stocks
stock_data = yf.download(all_stocks, start=start_date, progress=False,auto_adjust=True)['Close']


In [82]:


def generate_pairs(commodities_df:pd.DataFrame, stocks_df:pd.DataFrame, max_lag:int= 6) -> List[Tuple[pd.Series, pd.Series, int]]:
    """
    Generate all possible (commodity, stock, lag) combinations.
    
    Parameters:
        commodities_df: Input Dataframe of commodity closing prices with datetime index
        stocks_df: Input Dataframe of commodity closing prices with datetime index
        max_lag: Number of days for commodity lag 
    
    Returns:
        list: list of tuples of (commodity_series, stock_series) tuples
    """
    pairs = []
    for commodity, stock, lag in product(commodities_df.columns, stocks_df.columns, range(0,max_lag+1)):
        commodity_series = commodities_df[commodity]
        stock_series = stocks_df[stock]

        commodity_series.name = commodity
        stock_series.name = stock

        pairs.append((commodity_series,stock_series,lag))
    return pairs

def compute_features_for_pairs(pairs:List[Tuple[pd.Series,pd.Series]], rolling_window:int= 7, gradient:int= 5) -> Dict[str,pd.DataFrame]:
    """
    Compute features for multiple (commodity, stock) pairs. 
    
    Parameters:
        pairs: List of (commodity_series, stock_series) tuples
        rolling_window: Window size for rolling correlation
        gradient: Number of days for gradient computation 
    
    Returns:
        Dict[str, pd.DataFrame]: Dictionary with key=pair_name and value=feature_DataFrame
    """
    
    results = {}
    for commodity, stock, lag in pairs:
        commodity_name = f'{commodity.name}_lag_{str(lag)}'
        stock_name = stock.name
        pair_name = f"{commodity_name}_{stock_name}"
        
        commodity_returns =  np.log(commodity).diff(1+lag)
        stock_returns = np.log(stock).diff(1)
        
        
        df = pd.DataFrame(
            {'price': stock,
             commodity_name: commodity_returns,
            stock_name: stock_returns,
        }).dropna()
        
        df['rolling_corr'] = df[commodity_name].rolling(window=rolling_window).corr(df[stock_name])
        df['gradient'] = df[commodity_name].rolling(window=gradient).apply(lambda x: linregress(range(gradient), x).slope, raw=True)

        results[pair_name] = df.dropna()
    return results


def rank_gradients_across_pairs(features_dict:Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Create a DataFrame where each column is the gradient of a pair
    Rank gradients across all pairs for each day.

    Returns:
        pd.DataFrame with dates as index and ranks as values (lower=stronger gradient)
    """
    gradients = []
    for pair_id, df in features_dict.items():
        gradient_series = df['gradient'].copy()
        gradient_series.name = pair_id
        gradients.append(gradient_series)
    
    all_gradients_df = pd.concat(gradients,axis=1)
    ranked_gradients = all_gradients_df.abs().rank(axis=1,method='min',ascending=False)

    return ranked_gradients

def get_top_n_pairs_per_day(ranked_df:pd.DataFrame, n:int=10) -> pd.DataFrame:
    """
    Return top-N ranked pair IDs per day.
    
    Returns:
        pd.DataFrame with dates as index and top-N pair_ids as columns
    """
    return ranked_df.apply(lambda row: row.nsmallest(n).index.tolist(), axis=1).to_frame(name="top_pairs")


def detect_trade_signals(df,min_streak=3):
    """
    Operations:
        1. Identifies rolling correlation.
        2. Identifies streaks of outlier rolling correlations. - higher and lower quartiles
        3. Filters for top 50% absolute gradient values.
        4. Calucluates trade signals based on sign of the gradient.
    
    Parameters:
        df: Input Dataframe of commodity closing prices with datetime index
        thershold: int - threshold of rolling correlation
        min_streak: int - minimum number of consecutive days of correlations above threshold
    
    Returns:
        pd.DataFrame: dataframe with rolling_corr, gradient, and trade_signal
    """
    rolling_corr_quantiles = np.round(df.rolling_corr.quantile([.025,.975]).values,5)
    low_rolling_corr = min(-0.6,rolling_corr_quantiles[0])
    high_rolling_corr = max(0.6,rolling_corr_quantiles[1])
   
    
    df['over_thresh'] = df['rolling_corr'].apply(lambda x: 1 if x > high_rolling_corr or x < low_rolling_corr  else 0)  
    # Create a streak column that only counts consecutive days above/below thresholds
    df['streak'] = (
        df['over_thresh']
        .groupby(df['over_thresh'].ne(df['over_thresh'].shift()).cumsum())
        .cumcount() + 1
    ) * df['over_thresh']           # set to 0 when not over threshold

    # Filter for streaks that meet min_streak requirement
    gradient_quantiles = np.round(df[(df['streak']>=min_streak)]['gradient'].quantile([0.25,0.75]).values,5)
    low_gradient = gradient_quantiles[0]
    high_gradient = gradient_quantiles[1]
    
    df['signal'] = df.apply(
        lambda row: 1 if row['gradient'] > high_gradient and row['rolling_corr'] > high_rolling_corr and row['streak'] >= min_streak
        else (-1 if row['gradient'] < low_gradient and row['rolling_corr'] < low_rolling_corr and row['streak'] >= min_streak else 0),
        axis=1
    )

    return df

def n_day_return(signals, stocks):
    """
    Calcualates returns for buy and sells under a condition

        currently set to hold for (average_lag + streak)
        if buy signal, calculates return for buy
        if sell signal, calculates return for short
    
    Returns:
        List of Returns
    """
    # Make sure the index of stock_prices is datetime
    stock_data.index = pd.to_datetime(stock_data.index)
    # Create a list to store percent changes
    percent_changes = []

    average_streaks = signals[(signals.signal!=0)].groupby('pair')['streak'].mean().astype(int)
    ## Iterate through signals and stock data to calculate n day return from signal day
    for idx, row in signals.iterrows():
        if row['signal'] != 0:
            pair = row['pair'] # grab pair
            stock = row['stock_name'] # Grab stock name
            signal_date = idx.date().strftime("%Y-%m-%d")  # Grab date
            price_on_signal_day = row['price'] # Grab price on signal date
            
            hold_period = average_streaks[pair] + row['lag']  ## Holding for average streak + lag
            
            pos = stock_data.index.get_loc(signal_date)
            
            
            
            if pos + 3 < len(stock_data):
                price_n_days_later = stock_data.iloc[pos + hold_period][stock]
                if row['signal'] == 1:
                    percent_change = (price_n_days_later - price_on_signal_day) / price_on_signal_day
                elif row['signal'] == -1:
                    percent_change = (price_on_signal_day - price_n_days_later) / price_on_signal_day
                percent_changes.append(percent_change)
        else:
            percent_changes.append(None)

    return percent_changes

def detect_signals_from_top_pairs(top_3_per_day:pd.DataFrame, features:dict, min_streak:int = 3) -> pd.DataFrame:
    """
    Detect trade signals using top N ranked pairs per day and full signal logic.
    
    Parameters:
        top_3_per_day (pd.DataFrame): DataFrame with 'date' and 'pair' columns.
        features (dict): Dictionary of computed feature DataFrames for each pair.
        min_streak (int): Minimum streak length to trigger signal logic.

    Returns:
        pd.DataFrame: Signal DataFrame with date, pair, signal, gradient, rolling_corr, etc.
    """
    signal_rows = []
    for date, row in top_3_per_day.iterrows():
        for pair in row['top_pairs']:
            if pair not in features:
                continue

            feature_df = features[pair].copy()
            features_with_signals = detect_trade_signals(feature_df,min_streak=min_streak)

            if date in features_with_signals.index:
                signal_row = features_with_signals.loc[[date]].copy()
                signal_row['pair'] = pair
                signal_rows.append(signal_row[['pair','price','gradient','rolling_corr', 'signal','streak']])
    
    result_df = pd.concat(signal_rows).sort_index()
    result_df['lag'] = result_df['pair'].str.extract(r'lag_(\d+)').astype(int)
    result_df['stock_name'] = result_df['pair'].str.split('_').str[-1]

    result_df['n_day_return'] = n_day_return(result_df,stock_data)
    return result_df

In [91]:
pairs = generate_pairs(commodities_df=minerals, stocks_df=stock_data, max_lag=5)

In [92]:
features = compute_features_for_pairs(pairs,rolling_window=14, gradient=4)

In [93]:
rankings = rank_gradients_across_pairs(features)

In [94]:
top_3_per_day = get_top_n_pairs_per_day(rankings, n=10)

In [95]:
signals = detect_signals_from_top_pairs(top_3_per_day,features,min_streak=3)

In [96]:
print(f"Total Return Under Conditions: {signals[signals.signal!=0].n_day_return.sum()*100:.4g}%")

signals[signals.signal!=0]

Total Return Under Conditions: 39.16%


,pair,price,gradient,rolling_corr,signal,streak,lag,stock_name,n_day_return
Date,,,,,,,,,
2022-10-19,SI=F_lag_2_ADI,135.182861,0.027489,0.638296,1,3,2,ADI,0.000354
2022-10-19,SI=F_lag_2_AVGO,41.522522,0.027489,0.689491,1,4,2,AVGO,0.046175
2022-10-19,SI=F_lag_2_CMI,211.127365,0.027489,0.669275,1,4,2,CMI,0.063045
2023-03-15,PA=F_lag_4_AAPL,151.462524,0.042853,0.647571,1,3,4,AAPL,0.047454
2023-05-02,SB=F_lag_3_ADI,177.274979,-0.049281,-0.706453,-1,3,3,ADI,0.012142
2023-08-30,HE=F_lag_4_APD,282.549408,0.027141,0.645581,1,3,4,APD,0.024635
2023-09-18,PL=F_lag_5_AAPL,176.675110,0.027182,0.672217,1,3,5,AAPL,-0.040906
2023-10-24,HE=F_lag_4_AAPL,172.178040,0.069719,0.669515,1,3,4,AAPL,0.023812
2024-01-05,HE=F_lag_2_AVGO,103.221130,0.047216,0.644347,1,3,2,AVGO,0.055657


In [85]:
import backtester
import importlib
importlib.reload(backtester)
from backtester import Backtester
from backtester import Strategy

In [87]:
data = signals[['price','signal','pair']].dropna()

backtester.calculate_performance()

AttributeError: module 'backtester' has no attribute 'calculate_performance'

In [88]:
backtester.daily_portfolio_values

AttributeError: module 'backtester' has no attribute 'daily_portfolio_values'